**12.5. K-Fold Cross Validation**

*NOTE: This is a simplified version of the code used in 10_7_classification_hands_on.ipynb*

**1. Setup**

In [1]:
# installing the dependencies
%pip install numpy pandas scikit-learn -q


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

**2. Load Data**

Keep `daiabetes.csv` in the same folder as this notebook file

In [3]:
df = pd.read_csv("diabetes.csv")

In [4]:
df.shape

(768, 9)

In [5]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')

In [6]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


**⚕️ Pima Indians Diabetes Dataset**

| Column Name | Description |
|------------|-------------|
| Pregnancies | Number of times the patient has been pregnant |
| Glucose | Plasma glucose concentration after 2 hours in an oral glucose tolerance test |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Thickness of the triceps skin fold (mm) |
| Insulin | 2 hour serum insulin level (mu U/ml) |
| BMI | Body Mass Index (weight in kg / height in m²) |
| DiabetesPedigreeFunction | Likelihood of diabetes based on family history |
| Age | Age of the patient in years |
| Outcome | Target variable indicating diabetes status (1 = Diabetic, 0 = Non Diabetic) |

**3. Quick checks (high-level)**

We are not doing deep EDA / preprocessing for this video

In [7]:
# missing values
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [8]:
df["Outcome"].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [9]:
df["Outcome"].value_counts(normalize=True)*100

Outcome
0    65.104167
1    34.895833
Name: proportion, dtype: float64

**4. Split features and target**

In [10]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

In [11]:
X[:5]

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33


In [12]:
y[:5]

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

**5. K-Fold Setup**

In [13]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

**6. Loop through the folds**

In [14]:
a = [1, 2, 3, 4, 5]

for i, val in enumerate(a, start=1):
    print(f"Index - {i} : {val}")

Index - 1 : 1
Index - 2 : 2
Index - 3 : 3
Index - 4 : 4
Index - 5 : 5


In [15]:
# understanding folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    print(fold)
    print(len(train_idx))
    print(len(test_idx))
    print(train_idx[0:5])
    print(test_idx[0:5])
    print("-"*50)

1
614
154
[0 1 3 4 5]
[ 2  7 10 23 30]
--------------------------------------------------
2
614
154
[0 1 2 3 4]
[ 6 11 15 18 24]
--------------------------------------------------
3
614
154
[1 2 3 4 5]
[ 0  9 12 17 19]
--------------------------------------------------
4
615
153
[0 1 2 4 6]
[ 3  5  8 16 26]
--------------------------------------------------
5
615
153
[0 2 3 5 6]
[ 1  4 13 14 20]
--------------------------------------------------


In [16]:
# store metrics
accuracy_list = []
precision_list = []
recall_list = []
f1_list = []
roc_auc_list = []

In [17]:
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

    # split data
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # 7. scaling the data after splitting
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 8. train the model
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)

    # 9. predictions
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)

    # 10. metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred)

    # store metrics
    accuracy_list.append(acc)
    precision_list.append(prec)
    recall_list.append(rec)
    f1_list.append(f1)
    roc_auc_list.append(auc)

    # print fold result
    print(f"Fold {fold}")
    print(f"  Accuracy: {acc:.3f}")
    print(f"  Precision: {prec:.3f}")
    print(f"  Recall: {rec:.3f}")
    print(f"  F1 Score: {f1:.3f}")
    print(f"  ROC-AUC: {auc:.3f}")

    print("-"*40)

Fold 1
  Accuracy: 0.753
  Precision: 0.649
  Recall: 0.673
  F1 Score: 0.661
  ROC-AUC: 0.735
----------------------------------------
Fold 2
  Accuracy: 0.786
  Precision: 0.659
  Recall: 0.617
  F1 Score: 0.637
  ROC-AUC: 0.738
----------------------------------------
Fold 3
  Accuracy: 0.753
  Precision: 0.780
  Recall: 0.525
  F1 Score: 0.627
  ROC-AUC: 0.714
----------------------------------------
Fold 4
  Accuracy: 0.804
  Precision: 0.758
  Recall: 0.532
  F1 Score: 0.625
  ROC-AUC: 0.728
----------------------------------------
Fold 5
  Accuracy: 0.745
  Precision: 0.732
  Recall: 0.517
  F1 Score: 0.606
  ROC-AUC: 0.701
----------------------------------------


In [18]:
print(accuracy_list)
print(precision_list)

[0.7532467532467533, 0.7857142857142857, 0.7532467532467533, 0.803921568627451, 0.7450980392156863]
[0.6491228070175439, 0.6590909090909091, 0.7804878048780488, 0.7575757575757576, 0.7317073170731707]


In [19]:
np.mean(accuracy_list)

np.float64(0.7682454800101859)

In [20]:
mean_accuracy = round(np.mean(accuracy_list)*100, 2)
mean_precision = round(np.mean(precision_list)*100, 2)
mean_recall = round(np.mean(recall_list)*100, 2)
mean_f1_score = round(np.mean(f1_list)*100, 2)
mean_auc = round(np.mean(roc_auc_list)*100, 2)

In [21]:
print("K-FOld Cross Validation - Mean metrics:")
print(f"Mean Accuracy: {mean_accuracy} %")
print(f"Mean Precision: {mean_precision} %")
print(f"Mean Recall: {mean_recall} %")
print(f"Mean F1 Score: {mean_f1_score} %")
print(f"Mean ROC-AUC: {mean_auc} %")

K-FOld Cross Validation - Mean metrics:
Mean Accuracy: 76.82 %
Mean Precision: 71.56 %
Mean Recall: 57.27 %
Mean F1 Score: 63.13 %
Mean ROC-AUC: 72.33 %


**Key Takeaways**

- K-Fold Cross Validation evaluates a model using multiple train test splits.
- Each fold uses different data for testing, giving more reliable performance estimates.
- Always split the data before scaling to avoid data leakage.
- Fit the scaler only on training data and reuse it for test data.
- Use multiple metrics like Accuracy, Precision, Recall, F1-score, and ROC-AUC.
- ROC-AUC must be computed using predicted probabilities.
- Final model performance is the average of all fold scores.